# Aiscern — Text AI Detector Fine-Tuning
**Platform:** Google Colab (T4 GPU or CPU — both work)  
**Target:** `saghi776/aiscern-text-detector`  
**Base model:** `roberta-base`  
**Runtime:** ~30 minutes on T4, ~90 minutes on CPU  
**Expected accuracy:** >90% on HC3 benchmark

### Before running:
1. Runtime → Change runtime type → **T4 GPU** (optional but faster)
2. Click 🔑 key icon → Add secret: `HF_TOKEN`
3. Create `saghi776/aiscern-text-detector` repo on huggingface.co/new

### After training:
Wire the model into `hf-analyze.ts` by updating the text models:
```typescript
text_primary: 'saghi776/aiscern-text-detector',  // ← your fine-tuned model
```

In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers==4.40.0 datasets accelerate huggingface_hub \
    scikit-learn evaluate pandas

In [ ]:
# CELL 2 — Authenticate
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Authenticated with HuggingFace')

In [ ]:
# CELL 3 — Configuration
import torch

BASE_MODEL  = 'roberta-base'
REPO_ID     = 'saghi776/aiscern-text-detector'
MAX_LENGTH  = 512
BATCH_SIZE  = 16
EPOCHS      = 3
LR          = 2e-5
SEED        = 42
OUTPUT_DIR  = '/content/aiscern-text-detector'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# CELL 4 — Load HC3 dataset (gold standard for AI text detection)
# HC3 = Human ChatGPT Comparison Corpus: 37K question-answer pairs
# Each row has: human_answers[] and chatgpt_answers[]

from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd

print('Loading HC3 dataset...')
hc3 = load_dataset('Hello-SimpleAI/HC3', 'all', split='train')
print(f'HC3 rows: {len(hc3):,}')
print(f'Columns:  {hc3.column_names}')

# Flatten into (text, label) pairs
records = []
for row in hc3:
    # Human answers → label 0
    for h in (row.get('human_answers') or []):
        if h and len(h.strip()) > 80:
            records.append({'text': h.strip()[:3000], 'label': 0})
    # ChatGPT answers → label 1
    for a in (row.get('chatgpt_answers') or []):
        if a and len(a.strip()) > 80:
            records.append({'text': a.strip()[:3000], 'label': 1})

df = pd.DataFrame(records).sample(frac=1, random_state=SEED).reset_index(drop=True)
ai_count    = df['label'].sum()
human_count = len(df) - ai_count
print(f'\nFlattened: {len(df):,} samples — AI: {ai_count:,}  Human: {human_count:,}')

In [ ]:
# CELL 5 — Load additional datasets for diversity
# More AI sources = better generalisation to GPT-4/Claude/Gemini etc.

extra_records = []

# ChatGPT vs Real dataset
try:
    print('Loading ChatGPT-Real dataset...')
    ds = load_dataset('NicolaiSivesind/ChatGPT-Real', split='train')
    for row in ds:
        text = row.get('text') or row.get('content') or ''
        lbl  = row.get('label', row.get('generated', -1))
        if isinstance(lbl, bool): lbl = int(lbl)
        if text and len(text) > 80 and lbl in (0, 1):
            extra_records.append({'text': text[:3000], 'label': int(lbl)})
    print(f'  ✅ Added {len(extra_records):,} from ChatGPT-Real')
except Exception as e:
    print(f'  ⚠️  ChatGPT-Real skipped: {e}')

# Merge all
if extra_records:
    extra_df = pd.DataFrame(extra_records)
    df = pd.concat([df, extra_df], ignore_index=True).sample(frac=1, random_state=SEED)

# Balance classes
min_count = min(df['label'].value_counts())
df = pd.concat([
    df[df['label']==0].sample(min_count, random_state=SEED),
    df[df['label']==1].sample(min_count, random_state=SEED),
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'\nFinal balanced dataset: {len(df):,} samples ({min_count:,} per class)')

In [ ]:
# CELL 6 — Train / val / test split (80 / 10 / 10)
n        = len(df)
train_df = df.iloc[:int(n * 0.80)].reset_index(drop=True)
val_df   = df.iloc[int(n * 0.80):int(n * 0.90)].reset_index(drop=True)
test_df  = df.iloc[int(n * 0.90):].reset_index(drop=True)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

# Convert to HuggingFace Dataset
from datasets import Dataset
train_hf = Dataset.from_pandas(train_df)
val_hf   = Dataset.from_pandas(val_df)
test_hf  = Dataset.from_pandas(test_df)

In [ ]:
# CELL 7 — Tokenize
# id2label: 0='HUMAN', 1='AI'
# hf-analyze.ts: /ai|generated|artificial/i → matches 'AI'    ✅
#                /human|real|authentic/i    → matches 'HUMAN' ✅
from transformers import RobertaTokenizerFast

tokenizer = RobertaTokenizerFast.from_pretrained(BASE_MODEL)

def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )

print('Tokenizing...')
train_ds = train_hf.map(tokenize_batch, batched=True, desc='Train')
val_ds   = val_hf.map(tokenize_batch,   batched=True, desc='Val')
test_ds  = test_hf.map(tokenize_batch,  batched=True, desc='Test')

for ds in (train_ds, val_ds, test_ds):
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print('✅ Tokenization complete')

In [ ]:
# CELL 8 — Load RoBERTa model
from transformers import RobertaForSequenceClassification
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

model = RobertaForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'HUMAN', 1: 'AI'},
    label2id={'HUMAN': 0, 'AI': 1},
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable/1e6:.1f}M')

In [ ]:
# CELL 9 — Training
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='binary')),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=200,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    push_to_hub=True,
    hub_model_id=REPO_ID,
    hub_token=HF_TOKEN,
    hub_strategy='every_save',
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f'Training on {len(train_ds):,} samples for {EPOCHS} epochs...')
trainer.train()

In [ ]:
# CELL 10 — Evaluate
print('Evaluating on test set...')
test_results = trainer.evaluate(test_ds)

print(f"\n{'='*40}")
print(f"TEST ACCURACY: {test_results['eval_accuracy']:.4f}  ({test_results['eval_accuracy']*100:.1f}%)")
print(f"TEST F1:       {test_results['eval_f1']:.4f}")
print(f"{'='*40}")

if test_results['eval_accuracy'] >= 0.90:
    print('✅ Target accuracy >90% achieved!')
else:
    print('⚠️  Below 90% target — consider more epochs or data')

In [ ]:
# CELL 11 — Push to HuggingFace
commit_msg = (
    f"RoBERTa AI text detector — "
    f"accuracy={test_results['eval_accuracy']:.3f} "
    f"f1={test_results['eval_f1']:.3f} "
    f"(HC3 + ChatGPT-Real datasets)"
)
trainer.push_to_hub(commit_message=commit_msg)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'\n✅ Model pushed to: https://huggingface.co/{REPO_ID}')

In [ ]:
# CELL 12 — Verify output format + quick inference test
# hf-analyze.ts: /ai|generated|artificial/i → 'AI'    ✅
#                /human|real|authentic/i    → 'HUMAN' ✅
from transformers import pipeline

print('Verifying inference output...')
pipe = pipeline('text-classification', model=REPO_ID, token=HF_TOKEN,
                device=0 if torch.cuda.is_available() else -1)

# Test 1: obvious AI text
ai_text   = "Certainly! I'd be happy to help you understand the multifaceted landscape of artificial intelligence. At its core, AI encompasses a wide range of technologies that enable machines to perform tasks that typically require human intelligence."
# Test 2: obvious human text
human_text = "honestly i just wanna finish this report by friday. its not even that complicated but ive been putting it off all week lol. gonna make some coffee and just grind through it tonight"

for label, text in [('AI', ai_text), ('Human', human_text)]:
    result = pipe(text, truncation=True, max_length=512)
    print(f'\n[{label} sample]')
    print(f'Text: {text[:80]}...')
    print(f'Result: {result}')

print('\nExpected format: [{"label": "AI", "score": X}] or [{"label": "HUMAN", "score": X}]')

# Wire into hf-analyze.ts:
print('\n── To use this model in your app ──')
print('Update frontend/lib/inference/hf-analyze.ts:')
print(f'  text_primary: \'{REPO_ID}\',')
print('The model is now live and will be picked up automatically.')

In [ ]:
# CELL 13 — Optional: Wire text model into hf-analyze.ts
# ONLY run this if you want to update your codebase from Colab
# Requires your GitHub token

# Uncomment if you want to auto-update:
# import subprocess
# GH_TOKEN = userdata.get('GITHUB_TOKEN')
# 
# # Clone repo
# subprocess.run(['git', 'clone', f'https://{GH_TOKEN}@github.com/saghirahmed9067-png/DETECT-AI.git'])
# 
# # Update model ID
# hf_path = 'DETECT-AI/frontend/lib/inference/hf-analyze.ts'
# with open(hf_path) as f: content = f.read()
# content = content.replace(
#     "text_primary:   'openai-community/roberta-base-openai-detector'",
#     f"text_primary:   '{REPO_ID}',  // fine-tuned on HC3"
# )
# with open(hf_path, 'w') as f: f.write(content)
# 
# # Push
# subprocess.run(['git', '-C', 'DETECT-AI', 'add', '-A'])
# subprocess.run(['git', '-C', 'DETECT-AI', 'commit', '-m', f'feat: wire fine-tuned text model {REPO_ID}'])
# subprocess.run(['git', '-C', 'DETECT-AI', 'push'])
# print('✅ hf-analyze.ts updated and pushed to GitHub')

print('Text model fine-tuning complete!')
print(f'Model: https://huggingface.co/{REPO_ID}')